<a href="https://colab.research.google.com/github/fre-denni/genAIforUX-research/blob/main/logbook_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install libraries

In [ ]:
!pip install -q pymupdf pdf2image transformers==4.44.2 google-genai spacy pandas scikit-learn pillow openpyxl
!python -m spacy download en_core_web_sm
!pip install timm einops
!pip install flash_attn

We are using an older version of transformers from hugging face to be sure that it's compatible with the Florence2 model from microsoft

### Import necessary libraries

In [ ]:
import os
import fitz  # PyMuPDF
import pandas as pd
from google import genai
from google.genai import types
import spacy
from transformers import AutoProcessor, AutoModelForCausalLM
import transformers.dynamic_module_utils as dynamic_utils
from PIL import Image
import torch
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("Environment setup correctly!")

Setup folder



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Define the base folder
base_folder = "/content/drive/MyDrive/logbook analysis/"

# Define input and output folders
input_folder = os.path.join(base_folder, "logbook")
output_folder = os.path.join(base_folder, "output")
output_images_folder = os.path.join(base_folder, "output_images")

# Create output folders if they don't exist
os.makedirs(output_folder, exist_ok=True)
os.makedirs(output_images_folder, exist_ok=True)

print(f"Base folder: {base_folder}")
print(f"Input folder set to: {input_folder}")
print(f"Output folder created/verified: {output_folder}")
print(f"Output images folder created/verified: {output_images_folder}")


Add your API

In [ ]:
from google.colab import userdata
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API')
    client = genai.Client(api_key=GOOGLE_API_KEY)
    print("API configurata")
except Exception as e:
  print("Errore nel caricamento")
  print(f"errore: {e}")

Upload and define function of Florence-2

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Caricamento Florence-2 su: {device}...")

florence_model_id = "microsoft/Florence-2-large"
processor = AutoProcessor.from_pretrained(florence_model_id, trust_remote_code=True)

# TRUCCO: Bendiamo il parser di Hugging Face per fargli ignorare 'flash_attn'
original_get_imports = dynamic_utils.get_imports
def custom_get_imports(filename):
    imports = original_get_imports(filename)
    return [imp for imp in imports if imp != "flash_attn"]

dynamic_utils.get_imports = custom_get_imports # Applichiamo la benda

# Carichiamo il modello in sicurezza (userà l'algoritmo standard)
florence_model = AutoModelForCausalLM.from_pretrained(florence_model_id, trust_remote_code=True).to(device).eval()

dynamic_utils.get_imports = original_get_imports # Togliamo la benda (Best practice)

def execute_florence(task_prompt, image):
    inputs = processor(text=task_prompt, images=image, return_tensors="pt").to(device)
    generated_ids = florence_model.generate(
      input_ids=inputs["input_ids"],
      pixel_values=inputs["pixel_values"],
      max_new_tokens=1024,
      early_stopping=False,
      do_sample=False,
      num_beams=3,
    )
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed_answer = processor.post_process_generation(
        generated_text,
        task=task_prompt,
        image_size=(image.width, image.height)
    )
    return parsed_answer

from IPython.display import clear_output
clear_output()

print("Florence-2 ready!")

Configure spacy

In [ ]:
nlp = spacy.load("en_core_web_sm") # English language
ruler = nlp.add_pipe("entity_ruler", before="ner")

# Add custom rules for design methodologies
patterns = [
    {}
]

ruler.add_patterns(patterns)
print("Added custom config to spacy")

### Global function and prompts

Set up and force the JSON analysis for Gemini, and the various prompts for the tasks.

In [ ]:
# Global config of JSON
json_config = types.GenerateContentConfig(
    response_mime_type="application/json",
    temperature=0.1 # more deterministic outputs
)

def classify_assignment_gemini(text_snippet):
  prompt=
  response=

  return response.text

def analyze_schema_gemini(image_path):
  img=
  prompt=
  response=

  return response.text

def analyze_text_gemini(text_snippet):
  prompt=
  response=

  return response.text

def absa_analysis(text, entities):
  prompt=
  response=

  return response.text

